# Prepare data for prediction

In [1]:
# In case of problems with importing libraries
# Go to "Softwares" (blue cube symbol on the left)
# Load scipy-stack/2023a StdEnv/2020 gcc/9.3.0 root/6.26.06
# then re-select the Python 3.9 kernel on the top right
import numpy as np
from pathlib import Path
from multiprocessing import Pool, cpu_count
from tqdm import tqdm

import pandas as pd
from copy import deepcopy
import ROOT

import matplotlib.pyplot as plt
from matplotlib.ticker import AutoMinorLocator

# from calculatez import calc_z, calc_z_dist, calc_mu_for_wanted_z, calc_z_bump_scan, neg_log_likelihood

Welcome to JupyROOT 6.26/06


## Load functions

In [2]:
def get_readable_name(hname=None, selection=None, distribution=None):

    if 'hCat' not in hname:
        return hname

    # Objects definition
    objects = ['Wh', 'T', 'HM', 'Z', 'g', 'e', 'm', 'bExc', 'ex']
    extra = ['200met', 'incl']
    # Observable
    observables = {'mass' : 'm', 'massT' : 'mT', 'massMET' : 'mMET'}

    if hname:
        selection, distribution = hname.split('__')

    if hname or selection:

        elements = selection.split('_')[1:]
        non_zero_elements = [e for e in elements if e[0].isalpha() or (e[0].isdigit() and eval(e[0]) != 0)]

        # Identify objects
        present_objects = [e for e in non_zero_elements if e[1:] in objects or e[2:] in objects]
        final_state_objects = ''.join([p.replace('bExc','b').replace('ex','j') for p in present_objects])

        # Identify extra
        present_extra = [e for e in non_zero_elements if e[1:] not in objects and e[2:] not in objects]
        final_state_extra = '+'+'+'.join(present_extra)

        # Combine final state
        final_state = final_state_objects
        if len(present_extra) != 0:
            final_state += final_state_extra

    if hname or distribution:
        # Identify distribution observable
        obj = distribution.split('_')[0].replace('.',',')
        obs = distribution.split('_')[1]
        quantity = f'{observables[obs]}({obj})'


    if hname:
        out = final_state, quantity
    elif selection:
        out = final_state
    elif distribution:
        out = quantity
    return ' - '.join(out)

The history saving thread hit an unexpected error (OperationalError('database is locked')).History will not be written to the database.


## Load data

In [3]:
config = {
    'csv_file' : '/project/def-arguinj/shared/DDP_data/2023-02-21_DM_histograms_from_Samuel_v7/stlp_st1000_chan3/histos_Bmin30_Bmax200_thr0.csv',
    'name' : 'stlp_st1000_chan3_nomassT',
    'output_dir' : '/project/def-arguinj/shared/DDP_data/input_stlp_st1000_chan3_nomassT'
}

In [7]:
verbose = False

# Prepare output
outdir = f"{config['output_dir']}/{config['name']}"
Path(outdir).parent.mkdir(parents=True, exist_ok=True)
plots_dir = f'{outdir}/plots'
Path(plots_dir).parent.mkdir(parents=True, exist_ok=True)

# Load histograms from input
df_raw = pd.read_csv(config["csv_file"])
print(f'> Loaded {df_raw.shape[0]} histograms with'
      f' {df_raw["nbins"].min()} <= Nbins <= {df_raw["nbins"].max()} and'
      f' {df_raw["ymin"].min()} <= events per bin <= {df_raw["ymax"].max()}')

# Split selection and distribution information from the histogram name
df = deepcopy(df_raw)
df[['selection', 'distribution']] = df['Hname'].str.split('__', n=1, expand=True)

# Open ROOT file
root_file = config['csv_file'].replace('csv','root')
tfile = ROOT.TFile.Open(root_file ,"READ")

# Loop over histograms and save them
print(f'> Saving histograms...')
hists_data = {'hist' : [],
              'bins' : [], 
              'name' : []}

for i, row in tqdm(df.iterrows(), total=df.shape[0]):
    
    # Get selection and distribution
    selection = row['selection']
    distribution = row['distribution']
    if verbose:
        print(f'> Processing {row["Hname"]}...')

    # Get histogram entries
    columns_to_remove = ['Hname', 'nbins', 'ymin', 'ymax', 'selection', 'distribution']
    df_bin_entries = row.drop(columns_to_remove)
    df_bin_entries = df_bin_entries.loc[df_bin_entries != 0] # Delete bins with 0 entries
    h = df_bin_entries.to_numpy(dtype=np.int64)

    if verbose:
        print('Entries', len(h), h[:3], h[-3:])

    # Get binning with mass values
    th1 = tfile.Get(row['Hname'])
    if not th1: print("Failed to get histogram\n hist_name = %s\n root_file = %s" % (row['Hname'], root_file))
    th1.SetDirectory(0)
    bin_edges = np.array(list(th1.GetXaxis().GetXbins()))
    bin_edges = bin_edges[:h.shape[0]+1]
    bin_sizes = np.array([bin_edges[i+1]-bin_edges[i] for i in range(0, len(bin_edges)-1)])
    bin_centers = bin_edges[:-1] + bin_sizes/2
    if verbose:
        print('Mass edges', len(bin_edges), bin_edges[:3], bin_edges[-3:])
        print('Mass centers', len(bin_centers), bin_centers[:3], bin_centers[-3:])

    # Apply cuts, if applicable   
    # condition = True
    condition = 'massT' not in row['Hname']
    
    # Save all selected histograms that pass the selection
    if condition:
        hists_data['hist'].append(h.tolist()) # Save the histogram entries
        hists_data['bins'].append(list(bin_edges)) # Save the mass binning
        hists_data['name'].append(row['Hname']) # Save the histogram name

# Close ROOT file
tfile.Close()

# Get some stats
nbins, entries, mass = [], [], []
for hist, bins in zip(hists_data['hist'], hists_data['bins']):
    entries += hist
    mass += bins
    nbins += [len(hist)]
print(f'> Total of {len(hists_data["hist"])} histograms with\n'
      f'  {np.min(nbins)} <= Nbins <= {np.max(nbins)}\n'
      f'  {np.min(entries)} <= entries per bin <= {np.max(entries)}\n'
      f'  {np.min(mass)} <= mass <= {np.max(mass)}'
      )

# Save histograms into neural network input format
remove_NaN = lambda row : row[np.isfinite(row)]
hist = np.array([remove_NaN(np.array(row)) for row in hists_data['hist']], dtype=object)
bin_edges = np.array([remove_NaN(np.array(row)) for row in hists_data['bins']], dtype=object)
names = np.array([np.array([row]) for row in hists_data['name']], dtype=object)

for file_name, array in {'X':hist, 'M':bin_edges, 'C':names}.items():
    npy_file = f'{outdir}/{file_name}.npy'
    np.save(npy_file, array)
    print(f'> Saved {array.shape} histograms to {npy_file}')

> Loaded 7544 histograms with 30 <= Nbins <= 102 and 0.0 <= events per bin <= 156052.0
> Saving histograms...


100%|██████████| 7544/7544 [00:07<00:00, 1011.85it/s]


> Total of 7036 histograms with
  29 <= Nbins <= 101
  1 <= entries per bin <= 133274
  57.0 <= mass <= 7932.0
> Saved (7036,) histograms to /project/def-arguinj/shared/DDP_data/input_stlp_st1000_chan3_nomassT/stlp_st1000_chan3_nomassT/X.npy
> Saved (7036,) histograms to /project/def-arguinj/shared/DDP_data/input_stlp_st1000_chan3_nomassT/stlp_st1000_chan3_nomassT/M.npy
> Saved (7036, 1) histograms to /project/def-arguinj/shared/DDP_data/input_stlp_st1000_chan3_nomassT/stlp_st1000_chan3_nomassT/C.npy
